<a href="https://colab.research.google.com/github/SatyaSaiNath1311/Reinforcement-Learning/blob/main/LAB03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
GridWorld: Policy Evaluation and Value Iteration
Lab Assignment 3 - Reinforcement Learning Laboratory
"""

# ============================================================
# ENVIRONMENT DEFINITION
# ============================================================
GRID_SIZE = 4
START = (0, 0)
GOAL = (3, 3)
OBSTACLE = (1, 1)

ACTIONS = ['Up', 'Down', 'Left', 'Right']
ACTION_DELTA = {'Up': (-1, 0), 'Down': (1, 0), 'Left': (0, -1), 'Right': (0, 1)}
GAMMA = 0.9
STEP_REWARD = -1
GOAL_REWARD = 10

# State labels S1..S14 (row-major order, skipping the obstacle), Goal is terminal
STATE_ORDER = [(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE)
               if (r, c) != OBSTACLE and (r, c) != GOAL]
STATE_LABEL = {s: f"S{i+1}" for i, s in enumerate(STATE_ORDER)}
STATE_LABEL[GOAL] = "Goal"

def all_states():
    return [(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE) if (r, c) != OBSTACLE]

def is_terminal(s):
    return s == GOAL

def step(s, a):
    if is_terminal(s):
        return s, 0
    dr, dc = ACTION_DELTA[a]
    nr, nc = s[0] + dr, s[1] + dc
    if not (0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE) or (nr, nc) == OBSTACLE:
        nr, nc = s          # bounce back off wall / obstacle
    reward = GOAL_REWARD if (nr, nc) == GOAL else STEP_REWARD
    return (nr, nc), reward

STATES = all_states()

print("=" * 55)
print("TASK 1: GridWorld formulated as an MDP")
print("=" * 55)
print(f"States        : {len(STATES)} cells (excludes 1 obstacle at {OBSTACLE})")
print(f"Actions       : {ACTIONS}")
print(f"Reward        : {STEP_REWARD} per step, {GOAL_REWARD} on reaching the goal")
print(f"Terminal state: {GOAL}")
print(f"Start state   : {START}")

# ============================================================
# TASK 2: POLICY EVALUATION (equiprobable random policy)
# ============================================================
def equiprobable_policy(s):
    return {a: 0.25 for a in ACTIONS}

def policy_evaluation(policy_fn, theta=0.01, max_iter=5):
    V = {s: 0.0 for s in STATES}
    log = []
    for it in range(1, max_iter + 1):
        newV = V.copy()
        delta = 0
        for s in STATES:
            if is_terminal(s):
                continue
            v = sum(p * (step(s, a)[1] + GAMMA * V[step(s, a)[0]])
                    for a, p in policy_fn(s).items())
            newV[s] = v
            delta = max(delta, abs(v - V[s]))
        V = newV
        log.append((it, delta, "Converged" if delta < theta else "Not Converged"))
    return V, log

V_eval, log = policy_evaluation(equiprobable_policy)

print("\n" + "=" * 55)
print("TASK 2: Policy Evaluation (equiprobable random policy)")
print("=" * 55)
print(f"{'Iter':<6}{'Max Delta':<12}{'Status'}")
for it, d, status in log:
    print(f"{it:<6}{d:<12.4f}{status}")

# ============================================================
# TASK 3: VALUE ITERATION
# ============================================================
def value_iteration(theta=1e-4):
    V = {s: 0.0 for s in STATES}
    while True:
        delta = 0
        for s in STATES:
            if is_terminal(s):
                continue
            best = max(step(s, a)[1] + GAMMA * V[step(s, a)[0]] for a in ACTIONS)
            delta = max(delta, abs(best - V[s]))
            V[s] = best
        if delta < theta:
            break
    policy = {}
    for s in STATES:
        if is_terminal(s):
            policy[s] = '-'
            continue
        best_a, best_v = None, float('-inf')
        for a in ACTIONS:
            s2, r = step(s, a)
            v = r + GAMMA * V[s2]
            if v > best_v:
                best_v, best_a = v, a
        policy[s] = best_a
    return V, policy

V_star, pi_star = value_iteration()

print("\n" + "=" * 55)
print("TASK 3: Value Iteration - optimal policy and values")
print("=" * 55)
print(f"{'State':<12}{'Optimal Action':<16}{'State Value'}")
for s in STATE_ORDER + [GOAL]:
    label = f"{STATE_LABEL[s]} {s}"
    print(f"{label:<12}{pi_star[s]:<16}{V_star[s]:.2f}")

# ============================================================
# TASK 4: PATH COMPARISON
# ============================================================
import numpy as np

def naive_policy(s):
    r, c = s
    if c < GRID_SIZE - 1 and (r, c + 1) != OBSTACLE:
        return 'Right'
    return 'Down'

def run_policy(policy_type, max_steps=50, seed=0):
    rng = np.random.default_rng(seed)
    s = START
    path = [s]
    total_reward = 0
    for _ in range(max_steps):
        if is_terminal(s):
            return path, total_reward, True
        if policy_type == 'random':
            a = rng.choice(ACTIONS)
        elif policy_type == 'evaluated':
            a = naive_policy(s)
        else:
            a = pi_star[s]
        s2, r = step(s, a)
        total_reward += r
        s = s2
        path.append(s)
    return path, total_reward, is_terminal(s)

print("\n" + "=" * 55)
print("TASK 4: Path comparison across policies")
print("=" * 55)
print(f"{'Policy':<18}{'Path Length':<14}{'Total Reward':<14}{'Goal Reached'}")
for ptype, label in [('random', 'Random Policy'), ('evaluated', 'Evaluated Policy'), ('optimal', 'Optimal Policy')]:
    path, tot_r, reached = run_policy(ptype)
    print(f"{label:<18}{len(path)-1:<14}{tot_r:<14}{'Yes' if reached else 'No'}")
    if ptype != 'random':
        print("   path:", " -> ".join(str(p) for p in path))

TASK 1: GridWorld formulated as an MDP
States        : 15 cells (excludes 1 obstacle at (1, 1))
Actions       : ['Up', 'Down', 'Left', 'Right']
Reward        : -1 per step, 10 on reaching the goal
Terminal state: (3, 3)
Start state   : (0, 0)

TASK 2: Policy Evaluation (equiprobable random policy)
Iter  Max Delta   Status
1     1.7500      Not Converged
2     0.9000      Not Converged
3     0.8100      Not Converged
4     0.7290      Not Converged
5     0.6561      Not Converged

TASK 3: Value Iteration - optimal policy and values
State       Optimal Action  State Value
S1 (0, 0)   Down            1.81
S2 (0, 1)   Right           3.12
S3 (0, 2)   Down            4.58
S4 (0, 3)   Down            6.20
S5 (1, 0)   Down            3.12
S6 (1, 2)   Down            6.20
S7 (1, 3)   Down            8.00
S8 (2, 0)   Down            4.58
S9 (2, 1)   Down            6.20
S10 (2, 2)  Down            8.00
S11 (2, 3)  Down            10.00
S12 (3, 0)  Right           6.20
S13 (3, 1)  Right         